## Decision Graph
### Here we decide if a question needs retrieval or not

In [1]:
#dependencies
from typing import List, TypedDict, Literal
from pydantic import BaseModel, Field
import time
import os
from langchain_ollama import OllamaEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_groq import ChatGroq
from langchain_chroma import Chroma
from langchain_google_genai import  ChatGoogleGenerativeAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import OpenAIEmbeddings
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

load_dotenv()

C:\Users\Ritika Khandelwal\AppData\Local\Temp\ipykernel_27912\931874544.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


True

In [2]:
docs = (PyPDFLoader("./documents/Company_Policies.pdf").load()+ 
        PyPDFLoader("./documents/Company_Profile.pdf").load()+
        PyPDFLoader("./documents/Product_and_Pricing.pdf").load())

In [3]:
primary = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0)
fallback = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

llm = primary.with_fallbacks([fallback])

### Chunking and embedding

In [4]:
chunks = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=150
).split_documents(docs)

In [5]:
embeddings = OllamaEmbeddings(model="mxbai-embed-large")

persist_directory = "./chroma_db"

if os.path.exists(persist_directory):
    print("Loading existing vector database...")
    vector_store = Chroma(
        persist_directory=persist_directory,
        embedding_function=embeddings,
        collection_name="company_knowledge_base"
    )
else:
    print("Creating embeddings for the first time...")
    vector_store = Chroma.from_documents(
        documents=docs,
        embedding=embeddings,
        persist_directory=persist_directory,
        collection_name="company_knowledge_base"
    )

Loading existing vector database...


In [6]:
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

### Testing the Retriever

In [8]:
que = input('Enter your question')
ansr = retriever.invoke(que)
for d in ansr:
    print(d.page_content)
    print('-'*40)

Founder
Aarav Mehta founded NexaAI after over 10 years of experience in enterprise data platforms and
cloud infrastructure.
He previously worked with global consulting firms where he led multiple large-scale digital
transformation projects.
Leadership Team
The leadership team brings experience across AI engineering, product management, and business
operations.

Aarav Mehta – CEO & Founder

Riya Kapoor – CTO (Distributed systems & AI platforms)

Kunal Sharma – Head of Product

Neha Verma – Head of Operations & HR

Siddharth Rao – Head of Sales & Partnerships
----------------------------------------
NexaAI Solutions – Company Profile
Company Overview
NexaAI Solutions Pvt. Ltd. is a business-focused artificial intelligence company founded in 2021.
The company specializes in building enterprise-ready AI systems for knowledge management,
analytics, and automation.
NexaAI primarily serves mid-sized and large organizations across technology, finance, healthcare,
and education sectors.

